In [ ]:
# @title
import requests
import pandas as pd
import re
import time

# --- CONFIG ---
wiki_base_url = "https://en.wikipedia.org/w/api.php"
pause_between_requests = 1
# ----------------

movies = [
    {"title": "Atypical", "type": "tv", "year": 2017},
    {"title": "The Good Doctor", "type": "tv", "year": 2017},
    {"title": "Temple Grandin", "type": "film", "year": 2010},
    {"title": "Everything's Gonna Be Okay", "type": "tv", "year": 2020},
    {"title": "Taare Zameen Par", "type": "film", "year": 2007, "alt_titles": ["Like Stars on Earth"]},
    {"title": "Sitare Zameen Par", "type": "film", "year": 2025},
    {"title": "Front of the Class", "type": "film", "year": 2008},
    {"title": "The Tic Code", "type": "film", "year": 1998},
    {"title": "Patience", "type": "film", "year": 2017},
    {"title": "Barfi", "type": "film", "year": 2012},
    {"title": "My Name is Khan", "type": "film", "year": 2010, "alt_titles": ["MNIK"]},
    {"title": "Music", "type": "film", "year": 2021, "director": "Sia"},
    {"title": "Hichki", "type": "film", "year": 2018, "alt_titles": ["Hiccup"]},  # informal translation
    {"title": "Rain Man", "type": "film", "year": 1988},
    {"title": "Koi... Mil Gaya", "type": "film", "year": 2003, "alt_titles": ["Found Someone", "I Have Found Someone"]},
    {"title": "Extraordinary Attorney Woo", "type": "tv", "year": 2022, "alt_titles": ["Weird Lawyer Woo Young-woo"]}
]

neuro_terms = ["autism", "autistic", "neurodivergent", "Asperger", "ADHD",
               "representation", "portrayal", "spectrum"]

context_words = ['film', 'movie', 'series', 'show', 'character', 'portrays',
                 'depicts', 'stars', 'performance', 'actor', 'actress',
                 'review', 'watch', 'streaming', 'episode', 'season']

def mentions_title_with_boundary(text: str, title: str, alt_titles: list = None) -> bool:
    lower_text = text.lower()
    if len(title.split()) <= 2:
        pattern = r'\b' + re.escape(title.lower()) + r'\b'
        if re.search(pattern, lower_text):
            return True
    else:
        if title.lower() in lower_text:
            return True
    if alt_titles:
        for alt in alt_titles:
            if alt.lower() in lower_text:
                return True
    return False

def has_context_words(text: str) -> bool:
    lower_text = text.lower()
    return any(word in lower_text for word in context_words)

def count_mentions(text: str, title: str, alt_titles: list = None) -> int:
    lower_text = text.lower()
    count = 0
    if len(title.split()) <= 2:
        pattern = r'\b' + re.escape(title.lower()) + r'\b'
        count += len(re.findall(pattern, lower_text))
    else:
        start = 0
        while True:
            pos = lower_text.find(title.lower(), start)
            if pos == -1:
                break
            count += 1
            start = pos + 1
    if alt_titles:
        for alt in alt_titles:
            count += lower_text.count(alt.lower())
    return count

def fetch_wikipedia_talk_comments(movie_info: dict) -> list:
    title = movie_info["title"]
    alt_titles = movie_info.get("alt_titles", [])
    comments = []

    # Talk page title is usually "Talk:ArticleTitle"
    talk_page = f"Talk:{title}"

    params = {
        "action": "query",
        "format": "json",
        "titles": talk_page,
        "prop": "revisions",
        "rvprop": "content|timestamp|user",
        "rvslots": "main",
        "rvlimit": "max"
    }

    try:
        headers = {
      "User-Agent": "NeurodivergenceResearchBot/1.0 (your_email@example.com)"}
        resp = requests.get(wiki_base_url, params=params, headers=headers)
        if resp.status_code != 200:
            print(f"  API error {resp.status_code} for {title}")
            return comments

        pages = resp.json().get("query", {}).get("pages", {})
        for page_id, page_data in pages.items():
            revisions = page_data.get("revisions", [])
            for rev in revisions:
                text = rev.get("slots", {}).get("main", {}).get("*", "")
                if not text:
                    continue
                if not mentions_title_with_boundary(text, title, alt_titles):
                    continue
                if not has_context_words(text):
                    continue
                mention_count = count_mentions(text, title, alt_titles)
                comments.append({
                    "page_id": page_id,
                    "movie_title": title,
                    "movie_type": movie_info.get("type", "film"),
                    "user": rev.get("user"),
                    "timestamp": rev.get("timestamp"),
                    "text": text,
                    "mention_count": mention_count,
                    "source": "Wikipedia Talk"
                })

    except Exception as e:
        print(f"  Error fetching talk page for {title}: {e}")

    time.sleep(pause_between_requests)
    print(f"  Found {len(comments)} comments on Wikipedia Talk page for {title}")
    return comments

# --- Main Execution ---
all_wiki_comments = []
for movie in movies:
    comments = fetch_wikipedia_talk_comments(movie)
    all_wiki_comments.extend(comments)

df_wiki_comments = pd.DataFrame(all_wiki_comments)



In [ ]:
import requests
import pandas as pd
import re
import time

# --- CONFIG ---
wiki_base_url = "https://en.wikipedia.org/w/api.php"
pause_between_requests = 1
# ----------------

movies = [
    {"title": "Atypical", "type": "tv", "year": 2017},
    {"title": "The Good Doctor", "type": "tv", "year": 2017},
    {"title": "Temple Grandin", "type": "film", "year": 2010},
    {"title": "Everything's Gonna Be Okay", "type": "tv", "year": 2020},
    {"title": "Taare Zameen Par", "type": "film", "year": 2007, "alt_titles": ["Like Stars on Earth"]},
    {"title": "Sitare Zameen Par", "type": "film", "year": 2025},
    {"title": "Front of the Class", "type": "film", "year": 2008},
    {"title": "The Tic Code", "type": "film", "year": 1998},
    {"title": "Patience", "type": "film", "year": 2017},
    {"title": "Barfi", "type": "film", "year": 2012},
    {"title": "My Name is Khan", "type": "film", "year": 2010, "alt_titles": ["MNIK"]},
    {"title": "Music", "type": "film", "year": 2021, "director": "Sia"},
    {"title": "Hichki", "type": "film", "year": 2018, "alt_titles": ["Hiccup"]},
    {"title": "Rain Man", "type": "film", "year": 1988},
    {"title": "Koi... Mil Gaya", "type": "film", "year": 2003, "alt_titles": ["Found Someone", "I Have Found Someone"]},
    {"title": "Extraordinary Attorney Woo", "type": "tv", "year": 2022, "alt_titles": ["Weird Lawyer Woo Young-woo"]}
]

neuro_terms = ["autism", "autistic", "neurodivergent", "Asperger", "ADHD",
               "representation", "portrayal", "spectrum"]

context_words = ['film', 'movie', 'series', 'show', 'character', 'portrays',
                 'depicts', 'stars', 'performance', 'actor', 'actress',
                 'review', 'watch', 'streaming', 'episode', 'season']

# --- Text Cleaning Functions ---

def mentions_title_with_boundary(text: str, title: str, alt_titles: list = None) -> bool:
    lower_text = text.lower()
    if len(title.split()) <= 2:
        pattern = r'\b' + re.escape(title.lower()) + r'\b'
        if re.search(pattern, lower_text):
            return True
    else:
        if title.lower() in lower_text:
            return True
    if alt_titles:
        for alt in alt_titles:
            if alt.lower() in lower_text:
                return True
    return False

def has_context_words(text: str) -> bool:
    lower_text = text.lower()
    return any(word in lower_text for word in context_words)

def count_mentions(text: str, title: str, alt_titles: list = None) -> int:
    lower_text = text.lower()
    count = 0
    if len(title.split()) <= 2:
        pattern = r'\b' + re.escape(title.lower()) + r'\b'
        count += len(re.findall(pattern, lower_text))
    else:
        start = 0
        while True:
            pos = lower_text.find(title.lower(), start)
            if pos == -1:
                break
            count += 1
            start = pos + 1
    if alt_titles:
        for alt in alt_titles:
            count += lower_text.count(alt.lower())
    return count

def clean_wiki_text(text: str) -> str:
    # Remove nested templates {{...}}
    pattern = re.compile(r'\{\{[^{}]*\}\}')
    prev_text = None
    while prev_text != text:
        prev_text = text
        text = pattern.sub('', text)

    # Remove HTML comments and tags
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)
    text = re.sub(r'<[^>]+>', '', text)

    # Convert wiki links [[link|display]] or [[display]] -> "display"
    text = re.sub(r'\[\[([^|\]]*\|)?([^\]]+)\]\]', r'\2', text)

    # Remove user signatures and contribution links
    text = re.sub(r'\[\[User:[^\]]+\]\]', '', text)
    text = re.sub(r'\[\[Special:Contributions/[^\]]+\]\]', '', text)

    # Remove timestamps
    text = re.sub(r'\d{1,2}:\d{2}, \d{1,2} [A-Za-z]+ \d{4} \(UTC\)', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_user_comments_from_revision(text: str) -> list:
    """
    Splits revision text into individual user comments after cleaning.
    """
    text = clean_wiki_text(text)
    # Split by sections or typical end-of-comment markers
    comments = re.split(r'==.*?==', text)
    # Keep non-empty text sections
    comments = [c.strip() for c in comments if len(c.strip()) > 0]
    return comments

# --- Fetch Wikipedia Talk Comments ---

def fetch_wikipedia_talk_comments(movie_info: dict) -> list:
    title = movie_info["title"]
    alt_titles = movie_info.get("alt_titles", [])
    comments_list = []

    talk_page = f"Talk:{title}"

    params = {
        "action": "query",
        "format": "json",
        "titles": talk_page,
        "prop": "revisions",
        "rvprop": "content|timestamp|user",
        "rvslots": "main",
        "rvlimit": "max"
    }

    headers = {"User-Agent": "NeurodivergenceResearchBot/1.0 (your_email@example.com)"}

    try:
        resp = requests.get(wiki_base_url, params=params, headers=headers)
        if resp.status_code != 200:
            print(f"  API error {resp.status_code} for {title}")
            return comments_list

        pages = resp.json().get("query", {}).get("pages", {})
        for page_id, page_data in pages.items():
            revisions = page_data.get("revisions", [])
            for rev in revisions:
                text = rev.get("slots", {}).get("main", {}).get("*", "")
                if not text:
                    continue
                # Extract individual comments from the revision
                user_comments = extract_user_comments_from_revision(text)
                for c in user_comments:
                    if not mentions_title_with_boundary(c, title, alt_titles):
                        continue
                    if not has_context_words(c):
                        continue
                    mention_count = count_mentions(c, title, alt_titles)
                    comments_list.append({
                        "page_id": page_id,
                        "movie_title": title,
                        "movie_type": movie_info.get("type", "film"),
                        "user": rev.get("user"),
                        "timestamp": rev.get("timestamp"),
                        "text": c,
                        "mention_count": mention_count,
                        "source": "Wikipedia Talk"
                    })

    except Exception as e:
        print(f"  Error fetching talk page for {title}: {e}")

    time.sleep(pause_between_requests)
    print(f"  Found {len(comments_list)} comments on Wikipedia Talk page for {title}")
    return comments_list

# --- Main Execution ---
all_wiki_comments = []
for movie in movies:
    all_wiki_comments.extend(fetch_wikipedia_talk_comments(movie))

# Build DataFrame
df_wiki_comments = pd.DataFrame(all_wiki_comments)

# Optional: Remove duplicate comment text
df_wiki_comments = df_wiki_comments.drop_duplicates(subset=["text"])

# Preview
print(df_wiki_comments.head())
df_wiki_comments.to_csv("wikipedia_talk_comments_neurodivergence.csv", index=False)


  Found 12 comments on Wikipedia Talk page for Atypical
  Found 8 comments on Wikipedia Talk page for The Good Doctor
  Found 43 comments on Wikipedia Talk page for Temple Grandin
  Found 0 comments on Wikipedia Talk page for Everything's Gonna Be Okay
  Found 354 comments on Wikipedia Talk page for Taare Zameen Par
  Found 0 comments on Wikipedia Talk page for Sitare Zameen Par
  Found 5 comments on Wikipedia Talk page for Front of the Class
  Found 122 comments on Wikipedia Talk page for The Tic Code
  Found 73 comments on Wikipedia Talk page for Patience
  Found 0 comments on Wikipedia Talk page for Barfi
  Found 0 comments on Wikipedia Talk page for My Name is Khan
  Found 50 comments on Wikipedia Talk page for Music
  Found 11 comments on Wikipedia Talk page for Hichki
  Found 185 comments on Wikipedia Talk page for Rain Man
  Found 77 comments on Wikipedia Talk page for Koi... Mil Gaya
  Found 0 comments on Wikipedia Talk page for Extraordinary Attorney Woo
     page_id      movi

In [ ]:
from google.colab import files

files.download("wikipedia_talk_comments_neurodivergence.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>